# Numerical correlations

Correlates filtered numerical variables above the completeness threshold, identifies every highly correlated pair, and writes a formatted Excel report with a role-aware matrix and pair table.

**Project step:** §4.2 step 1.0.

**Inputs:** `outputs/03_phase_01_feature_funnel_numerical{suffix}.csv`, `outputs/03_phase_01_feature_selection_numerical{suffix}.csv`, `data/master_listing.parquet`, and `data/master_columns.csv`.

**Outputs:** `outputs/04_phase_01_correlations_numerical{suffix}.csv`, `outputs/04_phase_01_correlation_pairs{suffix}.csv`, and `outputs/04_phase_01_correlations_numerical{suffix}.xlsx`.

**Run after:** 03_feature_selection_numerical.

**Runtime:** ~5 min.


In [1]:
# ---- Data ----
INPUT = "data/master_listing.parquet"   # relative to the repo root
OUTPUT_DIR = "outputs"                  # every CSV / XLSX / PNG goes here

# ---- Filters (None = no filter) ----
CITY = None             # "chicago", "dfw", "dc", or a list of them       -> column `market`
NEIGHBORHOOD = None     # submarket/list -> column `market_building__submarket`
START_DATE = None       # "YYYY-MM-DD": keep listings with DATE_COLUMN >= START_DATE
END_DATE = None         # "YYYY-MM-DD": keep listings with DATE_COLUMN <= END_DATE
DATE_COLUMN = "market_unit__enter_market"

# ---- Correlations ----
MIN_COMPLETENESS = 70
METHOD = "spearman"
REDUNDANT = 0.9
TARGET = "market_price_observation__first_asking_rent"


In [2]:
from pathlib import Path

import pandas as pd

BLUE, GREEN, YELLOW, RED, BOLD, RESET = "\033[94m", "\033[92m", "\033[93m", "\033[91m", "\033[1m", "\033[0m"


def step(msg):
    print(f"\n{BOLD}{BLUE}▶ {msg}{RESET}")


def done(msg):
    print(f"{GREEN}  ✓ {msg}{RESET}")


def note(msg):
    print(f"{YELLOW}  • {msg}{RESET}")


ROOT = Path.cwd()
while not (ROOT / "data" / "master_listing.parquet").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
MASTER = ROOT / INPUT
CATALOG = ROOT / "data" / "master_columns.csv"
OUTPUTS = ROOT / OUTPUT_DIR
OUTPUTS.mkdir(parents=True, exist_ok=True)
NN = 4
FILTER_COLUMNS = ["market", "market_building__submarket", DATE_COLUMN]


def _as_list(value):
    """Normalize one filter value or a list of values."""
    if value is None:
        return None
    return [value] if isinstance(value, str) else list(value)


def suffix():
    """Build the shared output suffix from the active filters."""
    parts = []
    if CITY is not None:
        parts.append("+".join(map(str, _as_list(CITY))))
    if NEIGHBORHOOD is not None:
        parts.append("+".join(map(str, _as_list(NEIGHBORHOOD))))
    if START_DATE is not None or END_DATE is not None:
        parts.append(f"{START_DATE or 'start'}_to_{END_DATE or 'end'}")
    return "" if not parts else "__" + "__".join(parts)


def out(name, ext="csv"):
    """Build an output path with this notebook's number and active filters."""
    return OUTPUTS / f"{NN:02d}_phase_01_{name}{suffix()}.{ext}"


def upstream(number, name, ext="csv"):
    """Build an upstream notebook's output path with the active filters."""
    return OUTPUTS / f"{number:02d}_phase_01_{name}{suffix()}.{ext}"


def need(path, notebook):
    """Raise a clear instruction when a required upstream output is missing."""
    if not Path(path).exists():
        raise FileNotFoundError(f"Run {notebook} first; missing {path}")


def load(columns):
    """Read requested master columns and apply the configured listing filters."""
    requested = list(dict.fromkeys(columns))
    read_columns = list(dict.fromkeys(requested + FILTER_COLUMNS))
    frame = pd.read_parquet(MASTER, columns=read_columns)
    valid_values = {
        column: sorted(frame[column].dropna().astype(str).unique())
        for column in ("market", "market_building__submarket")
    }
    for column, value, label in (
        ("market", CITY, "city"),
        ("market_building__submarket", NEIGHBORHOOD, "neighborhood"),
    ):
        if value is not None:
            wanted = list(map(str, _as_list(value)))
            valid = valid_values[column]
            unknown = sorted(set(wanted) - set(valid))
            if unknown:
                raise ValueError(f"Unknown {label}(s) {unknown}. Valid values: {valid}")
            before = len(frame)
            frame = frame[frame[column].astype(str).isin(wanted)]
            note(f"{label}: {before:,} → {len(frame):,} rows")
    if START_DATE is not None:
        before = len(frame)
        frame = frame[frame[DATE_COLUMN] >= pd.Timestamp(START_DATE)]
        note(f"start date: {before:,} → {len(frame):,} rows")
    if END_DATE is not None:
        before = len(frame)
        frame = frame[frame[DATE_COLUMN] <= pd.Timestamp(END_DATE)]
        note(f"end date: {before:,} → {len(frame):,} rows")
    return frame[requested]


## Correlations and pairs

The matrix uses pairwise-complete filtered listings. The `redundant: keep one` flag applies only when both variables are candidates from notebook 03.

In [3]:
import numpy as np

FUNNEL_PATH = upstream(3, "feature_funnel_numerical")
SELECTION_PATH = upstream(3, "feature_selection_numerical")
need(FUNNEL_PATH, "03_feature_selection_numerical")
need(SELECTION_PATH, "03_feature_selection_numerical")
funnel = pd.read_csv(FUNNEL_PATH)
selection = pd.read_csv(SELECTION_PATH)
step(f"Keeping numerical variables more than {MIN_COMPLETENESS}% complete")
keep = funnel[funnel["completeness_pct"] > MIN_COMPLETENESS].copy()
n_complete = len(keep)
roles = keep.set_index("variable")["role"].to_dict()
roles[TARGET] = "target"
candidates = set(selection["feature"])
for role_name, count in keep["role"].value_counts().items():
    note(f"{role_name:9s}{count:>4}")
step(f"Loading {len(keep)} columns from {MASTER.name}")
df = load(keep["variable"].tolist())
done(f"{df.shape[0]:,} filtered rows x {df.shape[1]} columns")
constant = [column for column in df if df[column].nunique(dropna=True) <= 1]
if constant:
    df = df.drop(columns=constant)
    print(f"{RED}  ✗ dropped {len(constant)} constant column(s): {constant}{RESET}")
keep = keep[~keep["variable"].isin(constant)]
roles = {column: roles[column] for column in df.columns}
role = pd.Series(roles)
step(f"Computing {METHOD} correlations, pairwise complete")
corr = df.corr(method=METHOD)
corr.index.name = "variable"
done(f"{corr.shape[0]} x {corr.shape[1]} matrix")
step(f"Writing {out('correlations_numerical').relative_to(ROOT)}")
corr.round(4).to_csv(out("correlations_numerical"))
done(f"{corr.shape[0]} rows x {corr.shape[1]} columns")
step(f"Redundant pairs (|{METHOD}| >= {REDUNDANT})")
pairs = (corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack().dropna()
         .rename("corr").rename_axis(["var_1", "var_2"]).reset_index())
pairs["role_1"] = pairs["var_1"].map(roles)
pairs["role_2"] = pairs["var_2"].map(roles)
pairs = pairs.reindex(pairs["corr"].abs().sort_values(ascending=False).index).reset_index(drop=True)
redundant = pairs[pairs["corr"].abs() >= REDUNDANT]
is_x = redundant["var_1"].isin(candidates) & redundant["var_2"].isin(candidates)
done(f"{len(redundant)} of {len(pairs):,} pairs, {is_x.sum()} between two candidates")
for row in redundant[is_x].itertuples():
    print(f"{RED}  ✗ {row.corr:+.3f}{RESET}  {row.var_1}  ~  {row.var_2}")
note(f"the other {(~is_x).sum()} strong pairs involve the target, excluded, or non-X columns")
pairs.head(20)



▶ Keeping numerical variables more than 70% complete
  • feature    37
  • leaky      14
  • latent      8
  • label       7
  • inferred    1

▶ Loading 67 columns from master_listing.parquet


  ✓ 967,661 filtered rows x 67 columns


  ✗ dropped 1 constant column(s): ['truth_lease_event__inferred_term_months']

▶ Computing spearman correlations, pairwise complete


  ✓ 66 x 66 matrix

▶ Writing outputs/04_phase_01_correlations_numerical.csv
  ✓ 66 rows x 66 columns

▶ Redundant pairs (|spearman| >= 0.9)
  ✓ 122 of 2,145 pairs, 11 between two candidates
  ✗ +0.929  market_building__positive_staff_and_management  ~  market_building__positive_cleanliness_and_maintenance
  ✗ +0.923  market_unit__bed  ~  market_unit__sqft
  ✗ +0.920  market_building__positive_staff_and_management  ~  market_building__positive_amenities_and_features
  ✗ +0.920  market_building__positive_amenities_and_features  ~  market_building__positive_cleanliness_and_maintenance
  ✗ +0.917  market_building__count_reviews  ~  market_building__positive_cleanliness_and_maintenance
  ✗ +0.916  market_building__count_reviews  ~  market_building__positive_staff_and_management
  ✗ +0.911  market_building__count_reviews  ~  market_building__positive_amenities_and_features
  ✗ +0.909  market_building__positive_staff_and_management  ~  market_building__positive_location
  ✗ +0.906  market_bu

,var_1,var_2,corr,role_1,role_2
0,market_price_observation__last_effective_rent,market_price_observation__leased_effective_rent,1.000000,leaky,leaky
1,market_price_observation__last_asking_rent,market_price_observation__leased_asking_rent,1.000000,leaky,leaky
2,market_building__number_stories,truth_building_params__number_stories_true,1.000000,feature,latent
3,market_building__year_built,truth_building_params__year_built_true,1.000000,feature,latent
4,market_building__number_units,truth_building_params__number_units_true,1.000000,feature,latent
5,market_price_observation__last_asking_rent_psf,market_price_observation__leased_asking_rent_psf,1.000000,leaky,leaky
6,truth_lease_event__true_rent,market_price_observation__last_asking_rent,0.999773,label,leaky
7,truth_lease_event__true_rent,market_price_observation__leased_asking_rent,0.999773,label,leaky
8,truth_lease_event__true_effective_rent,market_price_observation__leased_effective_rent,0.999361,label,leaky
9,truth_lease_event__true_effective_rent,market_price_observation__last_effective_rent,0.999361,label,leaky


In [4]:
from datetime import date

from openpyxl import Workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter

REPORT = out("correlations_numerical", "xlsx")
NEG, MID, POS = "2166AC", "F7F7F7", "B2182B"   # blue = negative, white = none, red = positive
FONT = "Arial"
USABLE = {"feature", "inferred"}
ROLE_COLOR = {"feature": "000000", "inferred": "000000", "label": "B45F06",
              "leaky": "C00000", "latent": "7F7F7F", "target": "1E7B34"}
HEADER_FILL = PatternFill("solid", fgColor="D9D9D9")


def blend(v):
    """Hex color for a correlation: white (0) shading to blue (-1) or red (+1)."""
    end = POS if v >= 0 else NEG
    t = min(abs(v), 1)
    colors = []
    for index in (0, 2, 4):
        start_value = int(MID[index:index + 2], 16)
        end_value = int(end[index:index + 2], 16)
        colors.append(f"{round(start_value * (1 - t) + end_value * t):02X}")
    return "".join(colors)


def heatmap(cells):
    """Fill each cell with its color. Fills are written into the file rather than as Excel
    conditional formatting, so they also show in Numbers, Google Sheets and file previews.
    Dark cells get white text so the number stays readable; bold marks a redundant pair."""
    for cell in cells:
        v = cell.value
        cell.fill = PatternFill("solid", fgColor=blend(v))
        cell.font = Font(name=FONT, bold=abs(v) >= REDUNDANT, color="FFFFFF" if abs(v) >= 0.6 else "000000")


def header(ws, row=1):
    for cell in ws[row]:
        cell.font = Font(name=FONT, bold=True, color=cell.font.color)
        cell.fill = HEADER_FILL


wb = Workbook()
readme = wb.active
readme.title = "Read me"
matrix = wb.create_sheet("Matrix")
pairs_ws = wb.create_sheet("Pairs")

step("Matrix sheet: every variable vs every variable")
names = corr.columns.tolist()
n = len(names)
matrix.append(["variable", "role"] + names)
for v in names:
    matrix.append([v, role[v]] + corr.loc[v].round(4).tolist())
for j, cell in enumerate(matrix[1], 1):
    if j > 2:
        cell.font = Font(name=FONT, color=ROLE_COLOR[role[cell.value]])
        cell.alignment = Alignment(text_rotation=90, horizontal="center")
header(matrix)
for r in range(2, n + 2):
    for c in (1, 2):
        matrix.cell(r, c).font = Font(name=FONT, color=ROLE_COLOR[matrix.cell(r, 2).value])
    for c in range(3, n + 3):
        cell = matrix.cell(r, c)
        cell.font = Font(name=FONT)
        cell.number_format = "0.00"
        cell.alignment = Alignment(horizontal="center")
heatmap(c for row in matrix.iter_rows(min_row=2, min_col=3) for c in row)
matrix.row_dimensions[1].height = 290
matrix.column_dimensions["A"].width = max(map(len, names)) + 2
matrix.column_dimensions["B"].width = 9
for c in range(3, n + 3):
    matrix.column_dimensions[get_column_letter(c)].width = 5.5
matrix.freeze_panes = "C2"
done(f"{n} x {n} heatmap")

step("Pairs sheet: each pair once, strongest first")
p = pairs.assign(abs_corr=pairs["corr"].abs())
p["flag"] = np.where(p["abs_corr"] < REDUNDANT, "",
                     np.where(p["var_1"].isin(candidates) & p["var_2"].isin(candidates),
                              "redundant: keep one", "involves target / excluded / non-X column"))
pairs_ws.append(["variable_1", "role_1", "variable_2", "role_2", METHOD, f"|{METHOD}|", "flag"])
for r in p.itertuples():
    pairs_ws.append([r.var_1, r.role_1, r.var_2, r.role_2, round(r.corr, 4), round(r.abs_corr, 4), r.flag])
header(pairs_ws)
for row in pairs_ws.iter_rows(min_row=2):
    keep_one = row[6].value == "redundant: keep one"
    for cell in row:
        cell.font = Font(name=FONT, bold=keep_one)
    row[4].number_format = row[5].number_format = "0.000"
    if keep_one:
        row[6].fill = PatternFill("solid", fgColor="FFF2CC")
heatmap(row[0] for row in pairs_ws.iter_rows(min_row=2, min_col=5, max_col=5))
pairs_ws.auto_filter.ref = f"A1:G{len(p) + 1}"
pairs_ws.freeze_panes = "A2"
for col, width in zip("ABCDEFG", (max(map(len, names)) + 2, 9, max(map(len, names)) + 2, 9, 10, 10, 30)):
    pairs_ws.column_dimensions[col].width = width
done(f"{len(p):,} pairs, {(p['flag'] == 'redundant: keep one').sum()} flagged 'redundant: keep one'")

step("Read me sheet: method and color legend")
readme["A1"] = f"Correlation report: numerical variables more than {MIN_COMPLETENESS}% complete"
readme["A1"].font = Font(name=FONT, bold=True, size=14)
rows = [
    ("Source", f"data/master_listing.parquet, {len(df):,} listing episodes (one row per listing)"),
    ("Built by", f"notebooks/phase_01_market_rent/04_numerical_correlations.ipynb "
                f"on {date.today():%Y-%m-%d}"),
    ("Filters", f"city={CITY}; neighborhood={NEIGHBORHOOD}; start={START_DATE}; end={END_DATE}"),
    ("Variables", f"{n}: the {n_complete} numerical variables more than {MIN_COMPLETENESS}% complete, "
                  f"minus {len(constant)} constant column(s) ({', '.join(constant) or 'none'})"),
    ("Method", f"{METHOD.title()} correlation; each pair uses the rows where both variables are present"),
    ("Values", "Computed in Python and written as values (no formulas); "
               "same numbers as the correlations CSV."),
    ("", ""),
    ("Matrix", "Every variable vs every variable. Variable names are colored by role (see below)."),
    ("Pairs", "Each pair once, strongest first. Filter the 'flag' column on 'redundant: keep one'."),
    ("", ""),
    ("Cell colors", ""),
    ("-1.00", "strong negative: one goes up as the other goes down"),
    ("0.00", "no relationship"),
    ("+1.00", "strong positive: they go up together"),
    ("Bold", f"|correlation| >= {REDUNDANT}: the two variables carry the same information"),
    ("", ""),
    ("Name colors", ""),
    ("feature", "usable as X"),
    ("inferred", "usable as X (a platform inference available at prediction time)"),
    ("label", "a prediction target, not X"),
    ("target", "asking rent target, colored dark green; not X"),
    ("leaky", "NEVER in X: this listing's own advertised price"),
    ("latent", "NEVER in X: simulator internals, not observable in production"),
]
for label, text in rows:
    readme.append([label, text])
for row in readme.iter_rows(min_row=2):
    row[0].font = Font(name=FONT, bold=True, color=ROLE_COLOR.get(row[0].value, "000000"))
    row[1].font = Font(name=FONT)
    row[1].alignment = Alignment(wrap_text=True, vertical="top")
for label, fill, text_color in (("-1.00", NEG, "FFFFFF"), ("0.00", MID, "000000"), ("+1.00", POS, "FFFFFF")):
    cell = next(c for c in readme["A"] if c.value == label)
    cell.fill = PatternFill("solid", fgColor=fill)
    cell.font = Font(name=FONT, bold=True, color=text_color)
    cell.alignment = Alignment(horizontal="center")
readme.column_dimensions["A"].width = 14
readme.column_dimensions["B"].width = 110

wb.active = 1   # open on the Matrix sheet
REPORT.parent.mkdir(parents=True, exist_ok=True)
wb.save(REPORT)
done(f"wrote {REPORT.relative_to(ROOT)}")

pairs_csv = p[["var_1", "role_1", "var_2", "role_2", "corr", "abs_corr", "flag"]].copy()
pairs_csv.columns = ["variable_1", "role_1", "variable_2", "role_2", METHOD, f"|{METHOD}|", "flag"]
pairs_csv.to_csv(out("correlation_pairs"), index=False)
done(f"wrote {len(pairs_csv):,} pairs to {out('correlation_pairs').relative_to(ROOT)}")


▶ Matrix sheet: every variable vs every variable
  ✓ 66 x 66 heatmap

▶ Pairs sheet: each pair once, strongest first


  ✓ 2,145 pairs, 11 flagged 'redundant: keep one'

▶ Read me sheet: method and color legend
  ✓ wrote outputs/04_phase_01_correlations_numerical.xlsx
  ✓ wrote 2,145 pairs to outputs/04_phase_01_correlation_pairs.csv
